<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-09/NB09_unsupervised_signals_anomalies.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 09: Patterns without Labels: Clustering, Signals and Anomaly Detection

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook clusters the SEG 2016 well-log data with k-means after scaling and PCA and compares the clusters with the facies assigned by geologists [1]. It then retrieves earthquakes from the USGS catalogue for the region of Türkiye in 2023 and groups them with DBSCAN on the sphere [2, 3]. Finally it synthesises bearing vibration, computes spectra, envelope spectra and condition indicators, and trains an isolation forest on healthy windows to detect a developing fault [4].

This notebook is the hands-on part of Week 9. It opens with Python step 9: Signals, features and clustering with NumPy and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-09/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-09/Week09_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-09/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-09/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 9 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from scipy.stats import kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, roc_auc_score
from sklearn.ensemble import IsolationForest

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

## Python step 9: Signals, features and clustering with NumPy

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### Sampling a signal

A vibration sensor records values at a fixed sampling rate, for example 2000 samples per second. The time stamps follow from `np.arange(n) / rate`, and a sinusoid of frequency f is `np.sin(2 * np.pi * f * t)`. The practice signal below adds a component at the running speed of a shaft, a weaker component near a bearing defect frequency and random noise.

In [ ]:
import numpy as np

rate_hz = 2000                                    # samples per second
time_s = np.arange(2000) / rate_hz                # one second of time stamps
shaft = 0.8 * np.sin(2 * np.pi * 25 * time_s)     # 25 Hz running speed
fault = 0.3 * np.sin(2 * np.pi * 107 * time_s)    # component near a defect frequency
noise = 0.2 * np.random.default_rng(3).normal(size=time_s.size)
vib = shaft + fault + noise
print(time_s.size, time_s[:3], round(float(vib.std()), 3))

### Statistics of a signal

Condition monitoring starts with simple statistics. The root mean square (RMS) measures the energy of a signal, the peak is its largest absolute value, and the crest factor, the peak divided by the RMS, rises when short impacts appear, which is typical of developing bearing damage.

In [ ]:
rms = np.sqrt(np.mean(vib ** 2))
peak = np.max(np.abs(vib))
print(f"RMS {rms:.3f}, peak {peak:.3f}, crest factor {peak / rms:.2f}")

### Windows and the axis argument

Monitoring systems compute features for short windows of a signal. `reshape` rearranges the 2000 samples into a table of 10 windows with 200 samples each; the value -1 lets NumPy work out that length. Functions such as `np.mean` accept an `axis` argument: `axis=1` works along each row and returns one value per window, and `axis=0` works down the columns.

In [ ]:
windows = vib.reshape(10, -1)                   # 10 windows of 0.1 s each
rms_per_window = np.sqrt(np.mean(windows ** 2, axis=1))
print(windows.shape, rms_per_window.shape)
print(rms_per_window.round(3))

### Spectra

The fast Fourier transform decomposes a signal into sinusoids and shows at which frequencies its energy lies. For real signals, `np.fft.rfft` returns the non-negative frequencies, and `np.fft.rfftfreq` gives the frequency of each value. Dividing the magnitudes by half the number of samples turns them into amplitudes. A boolean mask restricts the search to a frequency band, and `np.isclose` compares floats with a tolerance instead of exact equality.

In [ ]:
spectrum = np.abs(np.fft.rfft(vib)) / (vib.size / 2)
freqs = np.fft.rfftfreq(vib.size, d=1 / rate_hz)
print(freqs[np.argmax(spectrum)], "Hz is the strongest component")
band = (freqs > 80) & (freqs < 150)
print(freqs[band][np.argmax(spectrum[band])], "Hz is the strongest component between 80 and 150 Hz")
print(f"amplitude at 107 Hz: {spectrum[np.isclose(freqs, 107)][0]:.3f}")

The envelope analysis of this week adds one step for bearing faults, whose short impacts spread their energy over many frequencies and appear clearly only after demodulation.

### Features for many windows at once

Unsupervised methods work on feature tables with one row per example. The code below simulates 20 windows of a healthy machine and 20 windows of a damaged one whose bearing produces short impacts. Broadcasting does most of the work: The sine wave of shape (200,) is added to noise of shape (20, 200), and NumPy repeats the sine for every row. Features computed along `axis=1` give one value per window, and `np.column_stack` joins them into a table.

In [ ]:
gen9 = np.random.default_rng(9)
t_w = np.arange(200) / rate_hz                              # windows of 0.1 s
healthy = 0.5 * np.sin(2 * np.pi * 25 * t_w) + 0.1 * gen9.normal(size=(20, 200))
damaged = 0.5 * np.sin(2 * np.pi * 25 * t_w) + 0.1 * gen9.normal(size=(20, 200))
damaged[:, ::40] += 1.5                                     # an impact every 20 ms
win = np.vstack([healthy, damaged])                         # 40 windows
rms_w = np.sqrt(np.mean(win ** 2, axis=1))
crest_w = np.max(np.abs(win), axis=1) / rms_w
F = np.column_stack([rms_w, crest_w])                       # feature table, one row per window
print(win.shape, F.shape)
print(F[[0, 1, 38, 39]].round(2))

### k-means with broadcasting

k-means alternates two steps: Every point joins its nearest centre, and every centre moves to the mean of its points. The features are first standardised column by column, so that RMS and crest factor count equally. The distances from all 40 points to both centres come from a single broadcast: `Z[:, None, :]` has shape (40, 1, 2), `centres[None, :, :]` has shape (1, 2, 2), and their difference has shape (40, 2, 2). Summing over the last axis gives a (40, 2) table of squared distances, and `argmin(axis=1)` picks the nearest centre for every point.

In [ ]:
Z = (F - F.mean(axis=0)) / F.std(axis=0)                    # standardise each column
centres = Z[[0, -1]].copy()                                 # start from the first and the last window
for _ in range(10):
    d = ((Z[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)
    cluster = d.argmin(axis=1)
    centres = np.array([Z[cluster == j].mean(axis=0) for j in range(2)])
print(d.shape, np.bincount(cluster))
print("clusters of the healthy windows:", cluster[:20])
print("clusters of the damaged windows:", cluster[20:])

### A simple anomaly flag

When examples of normal behaviour are available, a z-score flags values that lie far from them: z = (x - mean) / standard deviation, with the mean and the standard deviation taken from the healthy windows only. Windows with an absolute z above 3 are reported. The anomaly detection methods of this week extend the same idea to several features at once.

In [ ]:
mu, sd = rms_w[:20].mean(), rms_w[:20].std()
z = (rms_w - mu) / sd
print("flagged windows:", np.flatnonzero(np.abs(z) > 3))

**Quick check.** For a = np.arange(12).reshape(3, 4), what is the shape of a.sum(axis=0)?

A. (3,)  
B. (4,)  
C. (12,)  
D. (3, 4)

<details><summary>Show the answer</summary>

**B.** axis=0 sums down the rows, which leaves one value per column, and the array has four columns.

</details>

### Exercises for Python step 9

**Exercise 9.1.** Compute the RMS of `vib` and store it in `py_rms`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py9_vib():
    import numpy as np
    t = np.arange(2000) / 2000
    return 0.8 * np.sin(2 * np.pi * 25 * t) + 0.3 * np.sin(2 * np.pi * 107 * t) + 0.2 * np.random.default_rng(3).normal(size=t.size)


def _py9_rms_reference():
    import numpy as np
    v = _py9_vib()
    return float(np.sqrt(np.mean(v ** 2)))

In [ ]:
py_rms = None
check("Exercise 9.1", py_rms, _py9_rms_reference())

**Exercise 9.2.** Compute the crest factor of `vib` and store it in `py_crest`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py9_crest_reference():
    import numpy as np
    v = _py9_vib()
    return float(np.max(np.abs(v)) / np.sqrt(np.mean(v ** 2)))

In [ ]:
py_crest = None
check("Exercise 9.2", py_crest, _py9_crest_reference())

**Exercise 9.3.** Split `vib` into 20 windows of 0.05 s, compute the RMS of each window along the right axis, and store the position of the window with the largest RMS in `py_loudest_window`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py9_loudest_window_reference():
    import numpy as np
    w = _py9_vib().reshape(20, -1)
    return int(np.argmax(np.sqrt(np.mean(w ** 2, axis=1))))

In [ ]:
py_loudest_window = None
check("Exercise 9.3", py_loudest_window, _py9_loudest_window_reference())

**Exercise 9.4.** Count the windows that the z-score rule flags and store the count in `py_flagged`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py9_rms_windows():
    import numpy as np
    g = np.random.default_rng(9)
    tw = np.arange(200) / 2000
    h = 0.5 * np.sin(2 * np.pi * 25 * tw) + 0.1 * g.normal(size=(20, 200))
    d = 0.5 * np.sin(2 * np.pi * 25 * tw) + 0.1 * g.normal(size=(20, 200))
    d[:, ::40] += 1.5
    return np.sqrt(np.mean(np.vstack([h, d]) ** 2, axis=1))


def _py9_flagged_reference():
    import numpy as np
    r = _py9_rms_windows()
    return int((np.abs((r - r[:20].mean()) / r[:20].std()) > 3).sum())

In [ ]:
py_flagged = None
check("Exercise 9.4", py_flagged, _py9_flagged_reference())

**Exercise 9.5.** Store the shape of `Z[:, None, :] - centres[None, :, :]` as a tuple in `py_bshape`, first by reasoning about broadcasting and then by checking with `.shape`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py9_bshape_reference():
    return (40, 2, 2)

In [ ]:
py_bshape = None
check("Exercise 9.5", py_bshape, _py9_bshape_reference())

## 1. Well logs from the SEG 2016 machine learning contest

The data contain log measurements from nine wells in Kansas, sampled every half foot, together with a facies label assigned by geologists from cores [1]. Here the labels are hidden from the algorithms and used only afterwards for comparison. The photoelectric log PE is missing in some wells, so it is left out.

**Python move: reading a CSV file from a URL.** `pd.read_csv` accepts a web address as well as a file name.

In [ ]:
URL = "https://raw.githubusercontent.com/seg/2016-ml-contest/master/facies_vectors.csv"
LOGS = ["GR", "ILD_log10", "DeltaPHI", "PHIND", "NM_M", "RELPOS"]
try:
    wells = pd.read_csv(URL)
    WELL_SOURCE = "SEG 2016 facies data"
except Exception as error:
    print("Download failed, using a synthetic stand-in:", type(error).__name__)
    rng = np.random.default_rng(2016)
    centres = rng.normal(size=(9, len(LOGS))) * [25, 0.3, 5, 6, 0.5, 0.25] + [65, 0.65, 4, 13, 1.5, 0.5]
    lab = rng.integers(1, 10, 4000)
    wells = pd.DataFrame(centres[lab - 1] + rng.normal(size=(4000, len(LOGS))) * [12, 0.12, 3, 3, 0.2, 0.2], columns=LOGS)
    wells["Facies"] = lab
    WELL_SOURCE = "synthetic stand-in"
NAMES = {1: "SS", 2: "CSiS", 3: "FSiS", 4: "SiSh", 5: "MS", 6: "WS", 7: "D", 8: "PS", 9: "BS"}
Xw = StandardScaler().fit_transform(wells[LOGS])
print(WELL_SOURCE, Xw.shape)
wells["Facies"].map(NAMES).value_counts()

## 2. Principal components

In [ ]:
pca = PCA().fit(Xw)
Z = pca.transform(Xw)[:, :2]
print("explained variance ratio:", pca.explained_variance_ratio_.round(3))
pd.DataFrame(pca.components_[:2], columns=LOGS, index=["PC1", "PC2"]).round(2)

## 3. k-means and the number of clusters

**Python move: a loop that collects scores.** For k from 2 to 12, k-means with k-means++ initialisation [5] runs on the scaled logs, and the inertia and the silhouette on a subsample are recorded [6].

In [ ]:
ks, inertia, sil = range(2, 13), [], []
rng = np.random.default_rng(0)
sub = rng.choice(len(Xw), size=min(2000, len(Xw)), replace=False)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xw)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Xw[sub], km.labels_[sub]))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(list(ks), inertia, "o-"); ax[0].set_xlabel("k"); ax[0].set_ylabel("within-cluster sum of squares")
ax[1].plot(list(ks), sil, "o-"); ax[1].set_xlabel("k"); ax[1].set_ylabel("mean silhouette")
plt.tight_layout(); plt.show()

In [ ]:
km9 = KMeans(n_clusters=9, n_init=10, random_state=0).fit(Xw)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(Z[:, 0], Z[:, 1], c=wells["Facies"], cmap="tab10", s=4); ax[0].set_title("facies (geologists)")
ax[1].scatter(Z[:, 0], Z[:, 1], c=km9.labels_, cmap="tab10", s=4); ax[1].set_title("k-means clusters, k = 9")
for a in ax:
    a.set_xlabel("PC1"); a.set_ylabel("PC2")
plt.tight_layout(); plt.show()
print("adjusted Rand index between clusters and facies:", round(adjusted_rand_score(wells["Facies"], km9.labels_), 3))
pd.crosstab(wells["Facies"].map(NAMES), km9.labels_)

### Your turn, exercise 1

Compute the silhouette score of k-means with k = 3 (`n_init=10`, `random_state=0`) on the same subsample `Xw[sub]` and store it in `sil_k3`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _sil_k3_reference():
    km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Xw)
    return float(silhouette_score(Xw[sub], km.labels_[sub]))

In [ ]:
sil_k3 = None   # your computation
check("Exercise 1", sil_k3, _sil_k3_reference(), tol=1e-6)

## 4. Earthquakes around Türkiye in 2023

The USGS FDSN event service returns earthquake catalogues as CSV [2]. The query below asks for events of magnitude 4 or more in 2023 between 35 and 43 N and 25 and 45 E. DBSCAN with the haversine metric measures distances on the sphere in radians, so a radius of 25 km becomes 25 / 6371 [3]. The fallback creates a synthetic catalogue with an elongated aftershock zone and background events.

In [ ]:
Q = ("https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime=2023-01-01&endtime=2024-01-01"
     "&minlatitude=35&maxlatitude=43&minlongitude=25&maxlongitude=45&minmagnitude=4")
try:
    with urllib.request.urlopen(Q, timeout=30) as r:
        quakes = pd.read_csv(io.StringIO(r.read().decode()))
    QUAKE_SOURCE = "USGS catalogue, 2023"
except Exception as error:
    print("Download failed, using a synthetic catalogue:", type(error).__name__)
    rng = np.random.default_rng(6)
    t = rng.random(700)
    band = np.c_[36.3 + 2.0 * t + 0.08 * rng.normal(size=700), 36.1 + 2.4 * t + 0.08 * rng.normal(size=700)]
    aegean = np.c_[38.8 + 0.15 * rng.normal(size=60), 26.4 + 0.2 * rng.normal(size=60)]
    background = np.c_[rng.uniform(35, 43, 150), rng.uniform(25, 45, 150)]
    ll = np.vstack([band, aegean, background])
    quakes = pd.DataFrame({"latitude": ll[:, 0], "longitude": ll[:, 1], "mag": 4 + rng.exponential(0.4, len(ll))})
    QUAKE_SOURCE = "synthetic catalogue"
coords = np.radians(quakes[["latitude", "longitude"]].to_numpy())
db = DBSCAN(eps=25 / 6371, min_samples=10, metric="haversine", algorithm="ball_tree").fit(coords)
labels = db.labels_
print(QUAKE_SOURCE, len(quakes), "events;", len(set(labels) - {-1}), "clusters;", int(np.sum(labels == -1)), "noise events")
plt.figure(figsize=(7, 4))
plt.scatter(quakes["longitude"], quakes["latitude"], c=np.where(labels < 0, -1, labels), cmap="tab20", s=6 + 3 * (quakes["mag"] - 4) ** 2)
plt.xlabel("longitude"); plt.ylabel("latitude"); plt.title("DBSCAN clusters (dark blue = noise)"); plt.show()

### Your turn, exercise 2

Run DBSCAN again with a radius of 50 km and `min_samples=10`. Store the number of clusters, without the noise label -1, in `n_clusters_50km`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _n_clusters_50km_reference():
    lab = DBSCAN(eps=50 / 6371, min_samples=10, metric="haversine", algorithm="ball_tree").fit(coords).labels_
    return len(set(lab) - {-1})

In [ ]:
n_clusters_50km = None   # your computation
check("Exercise 2", n_clusters_50km, _n_clusters_50km_reference())

## 5. Bearing vibration: spectrum and envelope

The simulated bearing has the geometry of the 6205 bearing of the Case Western Reserve University benchmark: 9 balls and a ball-to-pitch diameter ratio of 0.2034 [7]. Each pass of a ball over an outer-race defect excites a 3 kHz resonance.

### Your turn, exercise 3

Compute the BPFO in hertz for a shaft speed of 1797 rpm and a contact angle of zero, and store it in `bpfo`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _bpfo_reference():
    return 9 / 2 * 1797 / 60 * (1 - 0.2034)

In [ ]:
bpfo = None   # your computation
check("Exercise 3", bpfo, _bpfo_reference(), tol=1e-6)

In [ ]:
FS = 12000


def bearing_signal(seconds=1.0, rpm=1797, severity=0.0, noise=0.5, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(int(seconds * FS)) / FS
    fr = rpm / 60
    x = 0.3 * np.sin(2 * np.pi * fr * t) + 0.5 * noise * rng.normal(size=t.size)
    if severity > 0:
        f_fault = 4.5 * fr * (1 - 0.2034)
        tau = np.arange(0, 0.04, 1 / FS)
        ring = severity * np.exp(-900 * tau) * np.sin(2 * np.pi * 3000 * tau)
        for t0 in np.arange(0.002, seconds, 1 / f_fault):
            i0 = int((t0 + 0.0002 * rng.normal()) * FS)
            seg = ring[: max(0, min(len(ring), t.size - i0))]
            x[i0:i0 + len(seg)] += seg
    return t, x


def spectrum(x):
    X = np.abs(np.fft.rfft(x * np.hanning(len(x)))) * 4 / len(x)
    return np.fft.rfftfreq(len(x), 1 / FS), X


t, x = bearing_signal(severity=0.6)
f, X = spectrum(x)
env = np.abs(hilbert(x))
fe, E = spectrum(env - env.mean())
fault = 4.5 * 1797 / 60 * (1 - 0.2034)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
ax[0].plot(t[:1200], x[:1200], lw=0.7); ax[0].set_xlabel("time (s)"); ax[0].set_title("waveform")
ax[1].plot(f, X, lw=0.7); ax[1].set_xlabel("frequency (Hz)"); ax[1].set_title("spectrum")
ax[2].plot(fe, E, lw=0.9); ax[2].set_xlim(0, 500); ax[2].set_xlabel("frequency (Hz)"); ax[2].set_title("envelope spectrum")
for m in range(1, 5):
    ax[2].axvline(m * fault, color="C3", ls="--", lw=0.8)
plt.tight_layout(); plt.show()

## 6. An anomaly detector trained on healthy data

Short windows of 0.1 s give four condition indicators: RMS, kurtosis, crest factor and the envelope amplitude near the BPFO. An isolation forest learns the healthy windows [4]; windows from a slowly growing fault are then scored.

In [ ]:
def indicators(x):
    rms = np.sqrt(np.mean(x ** 2))
    env = np.abs(hilbert(x)); fe, E = spectrum(env - env.mean())
    band = (fe > fault - 5) & (fe < fault + 5)
    return [rms, kurtosis(x), np.max(np.abs(x)) / rms, E[band].max()]


def windows(severity, n, seed):
    feats = []
    for k in range(n):
        _, x = bearing_signal(seconds=0.1, severity=severity, seed=seed + k)
        feats.append(indicators(x))
    return np.array(feats)


healthy = windows(0.0, 200, seed=0)
test_healthy = windows(0.0, 60, seed=1000)
SEVERITIES = [0.3, 0.5, 0.8, 1.2]
growing = np.vstack([windows(s, 20, seed=2000 + int(100 * s)) for s in SEVERITIES])
iso = IsolationForest(n_estimators=300, random_state=0).fit(healthy)
scores = -iso.score_samples(np.vstack([test_healthy, growing]))          # higher = more anomalous
truth = np.r_[np.zeros(len(test_healthy)), np.ones(len(growing))]
print("ROC AUC of the detector over all fault windows:", round(roc_auc_score(truth, scores), 3))
n0 = len(test_healthy)
for i, sev in enumerate(SEVERITIES):
    part = scores[n0 + 20 * i: n0 + 20 * (i + 1)]
    print(f"  severity {sev}: ROC AUC {roc_auc_score(np.r_[np.zeros(n0), np.ones(20)], np.r_[scores[:n0], part]):.3f}")
plt.figure(figsize=(7, 3)); plt.plot(scores, ".", ms=5); plt.axvline(len(test_healthy) - 0.5, color="k", ls="--", lw=0.8)
plt.xlabel("window (healthy, then growing fault)"); plt.ylabel("anomaly score"); plt.show()

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-09/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-09/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-09/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-09/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-09/README.md#research-and-report-assignment-optional) are listed in the week overview, and [Week 10](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-10/README.md) starts with its overview.

## References

[1] Hall, B. (2016). Facies classification using machine learning. *The Leading Edge*, *35*(10), 906-909. <https://doi.org/10.1190/tle35100906.1>

[2] U.S. Geological Survey (2026). *Earthquake Catalog API (FDSN Event Web Service)*. <https://earthquake.usgs.gov/fdsnws/event/1/>

[3] Ester, M., Kriegel, H.-P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. In *Proceedings of the Second International Conference on Knowledge Discovery and Data Mining (KDD-96)* (pp. 226-231). AAAI Press.

[4] Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). Isolation forest. In *2008 Eighth IEEE International Conference on Data Mining* (pp. 413-422). <https://doi.org/10.1109/ICDM.2008.17>

[5] Arthur, D., & Vassilvitskii, S. (2007). k-means++: The advantages of careful seeding. In *Proceedings of the Eighteenth Annual ACM-SIAM Symposium on Discrete Algorithms* (pp. 1027-1035). SIAM.

[6] Rousseeuw, P. J. (1987). Silhouettes: A graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics*, *20*, 53-65. <https://doi.org/10.1016/0377-0427(87)90125-7>

[7] Smith, W. A., & Randall, R. B. (2015). Rolling element bearing diagnostics using the Case Western Reserve University data: A benchmark study. *Mechanical Systems and Signal Processing*, *64-65*, 100-131. <https://doi.org/10.1016/j.ymssp.2015.04.021>